# `mboostrs` — base learner gallery

A **base learner** defines the form of one candidate effect in a boosting model. At each iteration, the algorithm selects the learner that best fits the current pseudo-residuals and adds a step of size `nu` in its direction.

The choice of base learner determines what kind of effect each predictor can have:

| Learner | Effect type |
|---|---|
| `bols` | Linear (ordinary least squares) |
| `bbs` | Smooth nonlinear (P-splines) |
| `bmono` | Monotone-constrained spline |
| `bns` | Natural spline basis, ridge-penalised |
| `bspatial` | 2D tensor-product smooth |
| `bmrf` | Markov random field (spatial smoothing) |
| `brandom` | Random effects (ridge-penalised factors) |
| `btree` | Decision tree stumps |
| `bkernel` | Kernel-ridge over an explicit kernel design |

This notebook fits each base learner to data suited to its effect type.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import polars as pl
import pandas as pd
import altair as alt
import mboostrs as mb

from mboostrs.datasets import bodyfat

rng = np.random.default_rng(42)
print("mboostrs version:", mb.__version__)

## 1. `bols` — ordinary least squares base learner

`bols(x)` fits a linear effect, $f(x) = \beta x$. `glmboost` uses this learner for its candidate effects.

In `gamboost`, you use `bols` explicitly when you want a linear effect for one predictor and a smooth effect (via `bbs`) for another.

**Data:** $y = 0.5 + 1.5 x_1 - 0.8 x_2 + \varepsilon$ with $\varepsilon \sim \mathcal{N}(0, 0.3^2)$.

In [ ]:
n = 300
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
y = 0.5 + 1.5 * x1 - 0.8 * x2 + rng.normal(0, 0.3, n)
data = pl.DataFrame({"x1": x1, "x2": x2, "y": y})

m_bols = mb.gamboost(
    "y ~ bols(x1) + bols(x2)", data=data,
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"Coefficients: {m_bols.coef()}")

## 1b. `bols` with `by=` — varying coefficient models

The `by=` argument creates a **varying-coefficient** model: the effect of $x$ is multiplied by $z$, i.e. $f(x, z) = \beta \cdot x \cdot z $. This lets the effect of one variable depend linearly on another.

**When to use:** interaction effects where you expect the slope of $x$ to change linearly with $z$. For non-linear interactions, combine with `bbs`.

In [ ]:
n = 400
x = rng.normal(size=n)
z = rng.uniform(0, 1, size=n)
y = 1.0 * x + 0.5 * z * x + rng.normal(0, 0.2, n)
data = pl.DataFrame({"x": x, "z": z, "y": y})

m_by = mb.gamboost(
    "y ~ bols(x, by=z) + bols(z)", data=data,
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"Coefficients: {m_by.coef()}")

## 2. `bbs` — P-spline smooth nonlinear effect

`bbs(x, df=..., knots=...)` builds a cubic B-spline basis with a difference penalty P-spline, following Eilers & Marx (1996). The `df` parameter controls the effective degrees of freedom — converted internally to a ridge penalty $\lambda$ via the Demmler-Reinsch identity (matching R's `mboost` exactly).

The `df` argument specifies the effective degrees of freedom; the implementation converts it to a penalty weight $\lambda$. Values near 1 produce nearly linear fits, while larger values allow more curvature.

**Data:** $y = \sin(2\pi x) + \varepsilon$, a classic nonlinear recovery problem.

**R equivalent:** `bbs(x, df=5, knots=25)`.

In [ ]:
n = 400
x = rng.uniform(0, 1, n)
y = np.sin(2 * np.pi * x) + rng.normal(0, 0.15, n)
data = pl.DataFrame({"x": x, "y": y})

m_bbs = mb.gamboost(
    "y ~ bbs(x, df=5, knots=25)", data=data,
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"Partial effects: {list(m_bbs.partial_effects().keys())}")
print(f"The bbs learner recovers the sin(2pi*x) shape from noisy data")

In [ ]:
# bbs fitted curve vs actual data
pe = m_bbs.partial_effects()
x_pe, y_pe = pe["x"]
pe_df = pl.DataFrame({"x": x_pe, "fitted": y_pe}).to_pandas()
data_pd = pl.DataFrame({"x": x, "y": y}).to_pandas()

alt.layer(
    alt.Chart(data_pd).mark_point(opacity=0.2, size=15).encode(
        x="x", y="y"
    ),
    alt.Chart(pe_df).mark_line(color="red", strokeWidth=3).encode(
        x="x", y="fitted:Q"
    ).properties(title="bbs(x, df=5, knots=25) \u2014 P-spline fit to sin(2\u03c0x)")
).interactive()


## 3. `bmono` — monotone-constrained P-spline

`bmono(x, constraint='increasing')` fits a P-spline that is constrained to be monotonically increasing, decreasing, or positive. This is useful when you have domain knowledge: e.g., the probability of default should increase with debt ratio.

**How it works:** the constrained fit uses iterative quadratic programming (QP) to project the unconstrained P-spline onto the space of monotone coefficients. This matches R's `mboost::bmono()`.

We demonstrate three constraints:
- **Increasing:** $y = \log(1 + e^{2x}) + \varepsilon$
- **Decreasing:** $y = -\log(1 + e^{2x}) + \varepsilon$
- **Positive:** $y = 0.5 + 1.5x^2 + \varepsilon$

In [ ]:
n = 200
x = np.sort(rng.uniform(-2, 2, n))

# --- Increasing constraint ---
y = np.log(1 + np.exp(2 * x)) + rng.normal(0, 0.2, n)
data = pl.DataFrame({"x": x, "y": y})

m_inc = mb.gamboost(
    "y ~ bmono(x, constraint='increasing', df=4, type='iterative')",
    data=data, control=mb.boost_control(mstop=150, nu=0.1),
)
print(f"Increasing constraint fit: mstop={m_inc.mstop}")

# --- Decreasing constraint ---
y_dec = -np.log(1 + np.exp(2 * x)) + rng.normal(0, 0.2, n)
data_dec = pl.DataFrame({"x": x, "y": y_dec})
m_dec = mb.gamboost(
    "y ~ bmono(x, constraint='decreasing', df=4)",
    data=data_dec, control=mb.boost_control(mstop=150, nu=0.1),
)
print(f"Decreasing constraint fit: mstop={m_dec.mstop}")

# --- Positive constraint ---
y_pos = 0.5 + 1.5 * x ** 2 + rng.normal(0, 0.1, n)
data_pos = pl.DataFrame({"x": x, "y": y_pos})
m_pos = mb.gamboost(
    "y ~ bmono(x, constraint='positive', df=4)",
    data=data_pos, control=mb.boost_control(mstop=150, nu=0.1),
)
print(f"Positive constraint fit: mstop={m_pos.mstop}")

In [ ]:
# Visualize all three monotone constraints
grid_sort = np.sort(x)
newdata = pl.DataFrame({"x": grid_sort})

inc_pe = m_inc.partial_effects()['x']
dec_pe = m_dec.partial_effects()['x']
pos_pe = m_pos.partial_effects()['x']

mono_data = []
for (pe, constraint, label) in [
    (inc_pe, "increasing", "inc"),
    (dec_pe, "decreasing", "dec"),
    (pos_pe, "positive", "pos"),
]:
    for xi, yi in zip(*pe):
        mono_data.append({"x": xi, "fitted": yi, "constraint": constraint})
mono_df = pd.DataFrame(mono_data)

raw_data = []
for xi, yi, constraint in [
    *[(x[i], y[i], "increasing") for i in range(len(x))],
    *[(x[i], y_dec[i], "decreasing") for i in range(len(x))],
    *[(x[i], y_pos[i], "positive") for i in range(len(x))],
]:
    raw_data.append({"x": xi, "y": yi, "constraint": constraint})
raw_df = pd.DataFrame(raw_data)

alt.layer(
    alt.Chart(mono_df).mark_line(strokeWidth=3).encode(
        x="x", y="fitted:Q", color="constraint:N"
    ).properties(title="bmono \u2014 monotone-constrained P-splines"),
    alt.Chart(raw_df).mark_point(opacity=0.15, size=10).encode(
        x="x", y="y", color="constraint:N"
    ),
).interactive()


## 4. `bns` — natural spline basis, ridge-penalised

`bns(x, df=..., knots=...)` fits a natural spline basis — a cubic spline with linear tails — with an **identity (ridge) penalty** on the coefficients. It differs from `bbs` in the penalty, not in having none: `bbs` penalises differences between adjacent coefficients, which shrinks toward a straight line, while `bns` shrinks every coefficient toward zero. Upstream `bns` is unpenalised, so a fit here is not the same object; `fit.parity_notes()` carries the deviation.

**When to use:** when you want a flexible fit without the shrinkage of P-splines, or when you need exact interpolation behaviour at the boundaries. In practice, `bbs` is usually preferred because the penalty protects against overfitting.

In [ ]:
n = 300
x = rng.uniform(0, 1, n)
y = 2 * x + np.sin(3 * np.pi * x) + rng.normal(0, 0.15, n)
data = pl.DataFrame({"x": x, "y": y})

m_bns = mb.gamboost(
    "y ~ bns(x, df=5, knots=15)", data=data,
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"bns fit: mstop={m_bns.mstop}")

## 5. `bspatial` — 2D tensor-product P-spline surface

`bspatial(x1, x2, df=..., knots=...)` builds a **tensor-product P-spline** — a 2D smooth surface defined by the Kronecker product of two B-spline bases, each with its own difference penalty.

**When to use:** spatial data, image analysis, or any setting where a response varies smoothly over a 2D coordinate space.

**Data:** $z = \sin(x_1) \cdot \cos(x_2) + \varepsilon$ over $[-2, 2] \times [-2, 2]$.

In [ ]:
n = 400
x1 = rng.uniform(-2, 2, n)
x2 = rng.uniform(-2, 2, n)
z = np.sin(x1) * np.cos(x2) + rng.normal(0, 0.1, n)
data = pl.DataFrame({"x1": x1, "x2": x2, "z": z})

m_sp = mb.gamboost(
    "z ~ bspatial(x1, x2, df=8, knots=12)",
    data=data, control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"bspatial fit: mstop={m_sp.mstop}")

## 6. `bmrf` — Markov random field (spatial smoothing)

`bmrf(region, K=penalty_matrix, df=...)` implements a Markov random field smoother. It requires an **adjacency matrix** $K$ (penalty matrix) where $K_{ij} \neq 0$ if regions $i$ and $j$ are neighbours.

**When to use:** areal spatial data — e.g., disease rates by county, election results by district. The MRF prior smooths estimates across neighbouring regions.

**Data:** 20 regions on a grid, with adjacency defined by geometric proximity. True region effects are generated from a spatial process, and we try to recover them.

**Note:** we construct the penalty matrix $K$ as the graph Laplacian: $L = D - W$ where $W$ is the adjacency matrix and $D$ the degree matrix.

In [ ]:
from mboostrs.learners import Bmrf

n = 200
n_regions = 20
region = rng.integers(0, n_regions, size=n).astype(float)

# Build adjacency matrix (grid adjacency)
W = np.zeros((n_regions, n_regions))
for i in range(n_regions):
    if i > 0:
        W[i, i - 1] = 1
    if i < n_regions - 1:
        W[i, i + 1] = 1
    if i > 3:
        W[i, i - 4] = 1
    if i < n_regions - 4:
        W[i, i + 4] = 1
np.fill_diagonal(W, 0)
W = ((W + W.T) > 0).astype(float)
K_penalty = np.diag(W.sum(axis=1)) - W  # Laplacian

true_effects = rng.normal(0, 0.5, size=n_regions)
y = true_effects[region.astype(int)] + rng.normal(0, 0.1, n)
data = pl.DataFrame({"region": region, "y": y})

lr_bmrf = Bmrf("region", K=K_penalty, df=5)
m_bmrf = mb.gamboost(
    "y ~ bmrf(region, df=5)", data=data,
    learners=[lr_bmrf],
    control=mb.boost_control(mstop=150, nu=0.1),
)
print(f"bmrf fit: mstop={m_bmrf.mstop}")

## 7. `brandom` — random effects (ridge-penalised factor)

`brandom(group, df=...)` fits a random intercept for each level of a factor using ridge (L2) penalisation. This is the boosting equivalent of a mixed model's random effect.

**When to use:** clustered/grouped data where observations within the same group are correlated. Examples: students within schools, patients within hospitals, repeated measures within subjects.

**Data:** 20 groups with group-specific intercepts drawn from $\mathcal{N}(0, 0.4^2)$, plus a common linear effect of $x$.

In [ ]:
n_groups = 20
n = 400
group = rng.integers(0, n_groups, size=n)
group_effects = rng.normal(0, 0.4, size=n_groups)
x = rng.normal(size=n)
y = 0.5 * x + group_effects[group] + rng.normal(0, 0.2, n)
data = pl.DataFrame({"x": x, "group": group.astype(float), "y": y})

m_re = mb.gamboost(
    "y ~ bols(x) + brandom(group, df=5)",
    data=data, control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"Random effects fit: mstop={m_re.mstop}")
print(f"Coefficients: {m_re.coef()}")

In [ ]:
# Random effects: estimated vs true group effects
coefs = m_re.coef()
est_effects = coefs.get("group", np.zeros(n_groups))
re_df = pd.DataFrame({"group": np.arange(n_groups),
                       "estimated": est_effects[:n_groups],
                       "true": group_effects})

chart = alt.Chart(re_df).mark_point(size=60, opacity=0.8).encode(
    x="true", y="estimated",
    tooltip=["group", "true", "estimated"],
).properties(title="brandom \u2014 random effects recovery")
chart
print(re_df[['group','true','estimated']].round(3).to_string(index=False))

## 8. `btree` — tree base learners (stumps)

`btree(x1, x2, maxdepth=...)` uses decision trees as base learners. With `maxdepth=1` you get stumps (the classic AdaBoost base learner); with larger depths you capture interactions.

Use `btree` when tree splits are appropriate for nonlinear effects or interactions. In this implementation, learners require finite input: a `NaN` raises an error. Set `na_action="omit"` to drop affected rows or impute them before fitting. `mboost` uses componentwise trees; tree-based boosting packages such as XGBoost, LightGBM, and CatBoost use different tree learners.

**Data:** a piecewise interaction surface with noise.

In [ ]:
n = 500
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
y = (
    0.5 * (x1 > 0) * (x2 > 0)
    - 0.3 * (x1 < 0) * (x2 > 0)
    + 0.2 * rng.normal(size=n)
)
data = pl.DataFrame({"x1": x1, "x2": x2, "y": y})

m_tree = mb.gamboost(
    "y ~ btree(x1, x2, maxdepth=2)",
    data=data, control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"btree fit: mstop={m_tree.mstop}")

## 9. `bkernel` — kernel smoothing

`bkernel(x, bandwidth=..., kernel='gaussian')` builds an explicit kernel design against the observed `x` values and fits it with a ridge penalty — kernel ridge regression, not the Nadaraya-Watson weighted average the name suggests, and not upstream's pathway/GWAS `bkernel`. `fit.parity_notes()` carries the deviation. The bandwidth controls the smoothness — larger = smoother.

**When to use:** when you want a local, non-parametric fit. Kernel methods excel at capturing local structure without global parametric assumptions.

Kernel options: `'gaussian'`, `'epanechnikov'`, `'biweight'`, `'triweight'`, `'tricube'`.

**Data:** $y = \sin(x) / (1 + 0.1 x^2) + \varepsilon$.

In [ ]:
n = 300
x = rng.uniform(-3, 3, n)
y = np.sin(x) / (1 + 0.1 * x ** 2) + rng.normal(0, 0.05, n)
data = pl.DataFrame({"x": x, "y": y})

# Gaussian kernel
m_kern = mb.gamboost(
    "y ~ bkernel(x, bandwidth=0.3, kernel='gaussian', df=5)",
    data=data, control=mb.boost_control(mstop=150, nu=0.1),
)
print(f"Gaussian kernel fit: mstop={m_kern.mstop}")

# Epanechnikov kernel (optimal in MSE sense)
m_kern2 = mb.gamboost(
    "y ~ bkernel(x, bandwidth=0.3, kernel='epanechnikov', df=5)",
    data=data, control=mb.boost_control(mstop=150, nu=0.1),
)
print(f"Epanechnikov kernel fit: mstop={m_kern2.mstop}")

## 10. Mixed learner models

A `gamboost` formula can combine learners with different effect forms. Choose a learner for each predictor based on the effect being modeled.

In this example using the **bodyfat** dataset:
- `bols(waistcirc)` — linear effect of waist circumference
- `bbs(hipcirc, df=4)` — smooth (potentially non-linear) effect of hip circumference
- `bols(elbowbreadth)` — linear effect of elbow breadth
- `bbs(kneebreadth, df=3)` — smooth effect of knee breadth

This boosted additive model combines linear terms and smooth effects.

In [ ]:
bf = bodyfat()
data_bf = bf
data_bf = data_bf.rename({"DEXfat": "y"})

m_mixed = mb.gamboost(
    "y ~ bols(waistcirc) + bbs(hipcirc, df=4) + bols(elbowbreadth) + bbs(kneebreadth, df=3)",
    data=data_bf, control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"Mixed bodyfat model: mstop={m_mixed.mstop}")
print(f"Coefficients: {m_mixed.coef()}")

# Inspect partial effects
pe = m_mixed.partial_effects()
for name in pe:
    print(f"  Partial effect: {name}, points={len(pe[name][1])}")

In [ ]:
# Partial effects for the mixed bodyfat model
pe = m_mixed.partial_effects()
parts = []
for name in pe:
    xv, yv = pe[name]
    for xi, yi in zip(xv, yv):
        parts.append({"x": xi, "partial_effect": yi, "term": name})
pe_df = pd.DataFrame(parts)

alt.Chart(pe_df).mark_line().encode(
    x="x", y="partial_effect:Q", color="term:N",
    row=alt.Row("term:N", title=None, header=alt.Header(labelAngle=0)),
).properties(title="Partial effects \u2014 bodyfat GAM", width=400)


## Summary

The base learners covered:

- **`bols`**: linear effects, simple and interpretable
- **`bols` with `by=`**: varying-coefficient / interaction models
- **`bbs`**: P-spline smooths — the most common non-linear learner
- **`bmono`**: monotone-constrained splines for domain knowledge
- **`bns`**: natural spline basis with a ridge penalty (upstream's is unpenalised — a recorded parity deviation)
- **`bspatial`**: 2D smooth surfaces
- **`bmrf`**: Markov random fields for areal spatial data
- **`brandom`**: random effects for clustered data
- **`btree`**: tree-based learners for automatic interactions
- **`bkernel`**: kernel smoothing for local fits

A single `gamboost` model can combine these learners to represent linear, smooth, spatial, grouped, and tree-based effects.